# RGAT L1: attention, message and gradient × attention heatmaps

This notebook replaces the former one-off plotting scripts.  It reads the three
root-level sparse exports in `runs_report/level1/rgat_l1_root_attention_all_relations/labeled`
and writes reproducible PNG, SVG, and cell-table CSV outputs to `figures/notebook_heatmaps`.

It creates two layouts for each source: (1) five exact forward kinship relations
(`child`, `spouse`, `sibling`, `father`, `mother`), and (2) the kinship/nonkinship
relation-family layout.  Attention and message each receive a raw-value view plus
a complete-budget share view; signed gradient × attention receives the two existing
attribution layouts.

**Metric choices.** Raw attention is `attention_mass`.  Raw message magnitude is
`absolute_message_l2_sum`, which adds the L2 magnitude of every edge contribution
and therefore does not hide magnitude through vector cancellation.  Its complete
root budget is `typed_absolute_message_l2_sum`.  This notebook deliberately keeps
the signed `gradient_x_attention` unchanged.

In [ ]:
from __future__ import annotations

import csv
import gzip
import math
from collections import defaultdict
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, Normalize, TwoSlopeNorm


# Edit only this cell for a different export or output location.
DATA_DIR = Path('runs_report/level1/rgat_l1_root_attention_all_relations/labeled').resolve()
OUTPUT_DIR = DATA_DIR / 'figures' / 'notebook_heatmaps'
OUTPUT_FORMATS = ('png', 'svg')
DPI = 240
SHOW_FIGURES = False  # Set True when exploring interactively.

L1_LABELS = ('Culture', 'Discovery/Science', 'Leadership', 'Other', 'Sports/Games')
FORWARD_KINSHIP = ('child', 'spouse', 'sibling', 'father', 'mother')
KINSHIP = frozenset((*FORWARD_KINSHIP, *(f'{relation}__rev' for relation in FORWARD_KINSHIP)))
FAMILIES = ('kinship', 'nonkinship')
EXPERIMENT = 'rgat_one_hop'
LAYER = '1'
SCORE = 'predicted-margin'

mpl.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titleweight': 'regular',
    'axes.labelcolor': '#202020',
    'text.color': '#202020',
})

# The same blue–white–red family as the supplied attribution figures.
SEQUENTIAL_CMAP = LinearSegmentedColormap.from_list(
    'rgat_sequential', ('#f7f7f7', '#d1e5f0', '#67a9cf', '#2166ac')
)
DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    'rgat_diverging', ('#b2182b', '#ef8a62', '#f7f7f7', '#67a9cf', '#2166ac')
)


def display_label(label: str) -> str:
    return label.replace('/', '/\n')


def relation_family(relation: str) -> str:
    return 'kinship' if relation in KINSHIP else 'nonkinship'


assert DATA_DIR.is_dir(), f'Missing data directory: {DATA_DIR}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Streaming aggregation

All three sparse exports are large.  The exporter emits all rows for a root together,
so the function below collapses visibility rows one root at a time instead of loading
the CSV into memory.  Raw matrices use the same conditional, pair-present mean as the
existing gradient × attention figures: sum matching rows within root, mean matching
roots within each seed, then equal-weight the seeds.  Family raw cells are weighted by
their matching root–relation count within each seed.

Budget-share matrices are different on purpose: their numerator includes zeros for
roots without a pair, and their denominator is the explicit complete root budget from
the matching roster.  Their cells therefore represent a share of all target-L1 budget.

In [ ]:
def open_csv(path: Path):
    if path.suffix == '.gz':
        return gzip.open(path, 'rt', newline='', encoding='utf-8')
    return path.open(newline='', encoding='utf-8')


def aggregate_sparse(sparse_path: Path, value_column: str, *, filter_score: bool = False):
    '''Return root-conditional statistics and complete-budget numerators.

    The raw-statistic keys are (seed, relation, source_l1, target_l1).  Budget
    numerators have the same key and are additive before any root normalization.
    '''
    raw_totals = defaultdict(float)
    raw_counts = defaultdict(int)
    budget_numerators = defaultdict(float)
    root_pairs = defaultdict(float)
    current_root = None
    accepted_rows = 0

    def flush_root():
        for key, value in root_pairs.items():
            raw_totals[key] += value
            raw_counts[key] += 1
        root_pairs.clear()

    with open_csv(sparse_path) as handle:
        for row in csv.DictReader(handle):
            if (
                row['experiment'] != EXPERIMENT
                or row['message_passing_layer'] != LAYER
                or row['source_l1'] not in L1_LABELS
                or row['target_l1'] not in L1_LABELS
                or (filter_score and row.get('score') != SCORE)
            ):
                continue

            root = (row['seed'], row['root_index'])
            if current_root is not None and root != current_root:
                flush_root()
            current_root = root

            key = (row['seed'], row['relation'], row['source_l1'], row['target_l1'])
            value = float(row[value_column])
            if not math.isfinite(value):
                raise ValueError(f'Non-finite {value_column} in {sparse_path} at root {root}')
            root_pairs[key] += value  # merges source_visibility within root
            budget_numerators[key] += value
            accepted_rows += 1

    flush_root()
    if not accepted_rows:
        raise ValueError(f'No matching rows found in {sparse_path}')
    return dict(raw_totals), dict(raw_counts), dict(budget_numerators)


def load_budget_totals(roster_path: Path, budget_column: str):
    '''Sum the complete root budget by (seed, target L1).'''
    totals = defaultdict(float)
    with open_csv(roster_path) as handle:
        for row in csv.DictReader(handle):
            if (
                row['experiment'] == EXPERIMENT
                and row['message_passing_layer'] == LAYER
                and row['target_l1'] in L1_LABELS
            ):
                value = float(row[budget_column])
                if not math.isfinite(value):
                    raise ValueError(f'Non-finite {budget_column} in {roster_path}')
                totals[(row['seed'], row['target_l1'])] += value
    if not totals:
        raise ValueError(f'No matching root budgets found in {roster_path}')
    return dict(totals)


def average_seed_values(per_seed: dict, keys, seeds, *, missing_is_zero: bool):
    cells = {}
    for key in keys:
        values = []
        for seed in seeds:
            seed_key = (seed, *key)
            if seed_key in per_seed:
                values.append(per_seed[seed_key])
            elif missing_is_zero:
                values.append(0.0)
        cells[key] = float(np.mean(values)) if values else float('nan')
    return cells


def raw_cells(raw_totals: dict, raw_counts: dict):
    '''Build conditional raw-value cells for exact relations and families.'''
    seeds = sorted({seed for seed, _relation, _source, _target in raw_totals})
    exact_per_seed = {
        key: raw_totals[key] / raw_counts[key]
        for key in raw_totals
    }
    exact_keys = [
        (relation, source, target)
        for relation in FORWARD_KINSHIP
        for source in L1_LABELS
        for target in L1_LABELS
    ]
    exact = average_seed_values(exact_per_seed, exact_keys, seeds, missing_is_zero=False)

    family_total = defaultdict(float)
    family_count = defaultdict(int)
    for (seed, relation, source, target), total in raw_totals.items():
        family_key = (seed, relation_family(relation), source, target)
        family_total[family_key] += total
        family_count[family_key] += raw_counts[(seed, relation, source, target)]
    family_per_seed = {
        key: family_total[key] / family_count[key]
        for key in family_total
    }
    family_keys = [
        (family, source, target)
        for source in L1_LABELS
        for family in FAMILIES
        for target in L1_LABELS
    ]
    family = average_seed_values(family_per_seed, family_keys, seeds, missing_is_zero=False)
    return exact, family


def budget_share_cells(budget_numerators: dict, budget_totals: dict):
    '''Build complete-budget percent-share cells for exact relations and families.'''
    seeds = sorted({seed for seed, _target in budget_totals})
    exact_keys = [
        (relation, source, target)
        for relation in FORWARD_KINSHIP
        for source in L1_LABELS
        for target in L1_LABELS
    ]
    exact_per_seed = {}
    for seed in seeds:
        for relation, source, target in exact_keys:
            denominator = budget_totals[(seed, target)]
            exact_per_seed[(seed, relation, source, target)] = (
                100.0 * budget_numerators.get((seed, relation, source, target), 0.0) / denominator
            )
    exact = average_seed_values(exact_per_seed, exact_keys, seeds, missing_is_zero=True)

    family_numerators = defaultdict(float)
    for (seed, relation, source, target), value in budget_numerators.items():
        family_numerators[(seed, relation_family(relation), source, target)] += value
    family_keys = [
        (family, source, target)
        for source in L1_LABELS
        for family in FAMILIES
        for target in L1_LABELS
    ]
    family_per_seed = {}
    for seed in seeds:
        for family, source, target in family_keys:
            denominator = budget_totals[(seed, target)]
            family_per_seed[(seed, family, source, target)] = (
                100.0 * family_numerators.get((seed, family, source, target), 0.0) / denominator
            )
    family = average_seed_values(family_per_seed, family_keys, seeds, missing_is_zero=True)
    return exact, family


def cells_dataframe(cells: dict, layout: str, metric: str) -> pd.DataFrame:
    rows = []
    if layout == 'exact_kinship':
        for relation in FORWARD_KINSHIP:
            for source in L1_LABELS:
                for target in L1_LABELS:
                    rows.append({
                        'relation': relation,
                        'source_l1': source,
                        'target_l1': target,
                        metric: cells[(relation, source, target)],
                    })
    elif layout == 'relation_family':
        for source in L1_LABELS:
            for family in FAMILIES:
                for target in L1_LABELS:
                    rows.append({
                        'relation_family': family,
                        'source_l1': source,
                        'target_l1': target,
                        metric: cells[(family, source, target)],
                    })
    else:
        raise ValueError(f'Unknown layout: {layout}')
    return pd.DataFrame(rows)

## Shared figure style

Non-negative attention/message matrices use white-to-blue.  The signed attribution
matrices use red–white–blue with white fixed at zero.  The grid, row grouping,
annotation weights, and colourbar placement intentionally match the two supplied
`gradient × attention` examples.

In [ ]:
def nice_positive_limit(values) -> float:
    maximum = max(float(value) for value in values if np.isfinite(value))
    if maximum == 0:
        return 1.0
    magnitude = 10 ** math.floor(math.log10(maximum))
    for multiplier in (1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0):
        candidate = multiplier * magnitude
        if maximum <= candidate:
            return candidate
    raise AssertionError('Unreachable')


def nice_symmetric_limit(values) -> float:
    maximum = max(abs(float(value)) for value in values if np.isfinite(value))
    return nice_positive_limit((maximum,)) if maximum else 0.01


def matrix_for(cells: dict, layout: str):
    if layout == 'exact_kinship':
        row_keys = [(relation, source) for relation in FORWARD_KINSHIP for source in L1_LABELS]
        matrix = np.array([
            [cells[(relation, source, target)] for target in L1_LABELS]
            for relation, source in row_keys
        ])
    elif layout == 'relation_family':
        row_keys = [(family, source) for source in L1_LABELS for family in FAMILIES]
        matrix = np.array([
            [cells[(family, source, target)] for target in L1_LABELS]
            for family, source in row_keys
        ])
    else:
        raise ValueError(f'Unknown layout: {layout}')
    return row_keys, matrix


def plot_heatmap(
    cells: dict,
    *,
    layout: str,
    title: str,
    colorbar_label: str,
    footer: str,
    filename: str,
    signed: bool,
    value_format,
):
    row_keys, matrix = matrix_for(cells, layout)
    values = matrix.ravel()
    if signed:
        limit = nice_symmetric_limit(values)
        norm = TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit)
        cmap = DIVERGING_CMAP
    else:
        limit = nice_positive_limit(values)
        norm = Normalize(vmin=0.0, vmax=limit)
        cmap = SEQUENTIAL_CMAP

    exact = layout == 'exact_kinship'
    figure, axis = plt.subplots(figsize=(13.2, 17.6 if exact else 9.2))
    figure.subplots_adjust(left=0.27 if exact else 0.31, right=0.82, top=0.92, bottom=0.10)
    image = axis.imshow(matrix, cmap=cmap, norm=norm, aspect='auto')

    rows, columns = matrix.shape
    for row in range(rows + 1):
        axis.axhline(row - 0.5, color='white', linewidth=0.9, zorder=3)
    for column in range(columns + 1):
        axis.axvline(column - 0.5, color='white', linewidth=0.9, zorder=3)
    block_size = len(L1_LABELS) if exact else 2
    for divider in range(block_size, rows, block_size):
        axis.axhline(divider - 0.5, color='#535353', linewidth=1.4, zorder=4)
    axis.add_patch(plt.Rectangle((-0.5, -0.5), columns, rows, fill=False, edgecolor='#202020', linewidth=1.2, zorder=5))

    for row, key in enumerate(row_keys):
        if exact:
            relation, source = key
            axis.text(-0.62, row, source, ha='right', va='center', fontsize=10.5, clip_on=False)
            if row % len(L1_LABELS) == 0:
                axis.text(-1.72, row + (len(L1_LABELS) - 1) / 2, relation, ha='right', va='center', fontsize=11.5, fontweight='medium', clip_on=False)
        else:
            family, source = key
            axis.text(-0.62, row, family, ha='right', va='center', fontsize=10.5, clip_on=False)
            if family == 'kinship':
                axis.text(-1.72, row + 0.5, display_label(source), ha='right', va='center', fontsize=11.5, fontweight='medium', clip_on=False)

        for column, value in enumerate(matrix[row]):
            if not np.isfinite(value):
                label, color = '—', '#202020'
            else:
                label = value_format(value)
                strength = abs(value) / limit if signed else value / limit
                color = 'white' if strength >= 0.56 else '#202020'
            axis.text(column, row, label, ha='center', va='center', fontsize=10.5, color=color)

    axis.set_xticks(range(columns), [display_label(label) for label in L1_LABELS], fontsize=11)
    axis.xaxis.tick_top()
    axis.tick_params(axis='x', length=0, pad=10)
    axis.set_yticks([])
    axis.set_xlim(-2.18 if exact else -3.4, columns - 0.5)
    axis.set_ylim(rows - 0.5, -0.5)
    axis.set_xlabel('Target labeled-root L1', fontsize=12, labelpad=18)
    axis.set_title(title, loc='left', fontsize=16.5, pad=58)
    if not exact:
        axis.text(0.035, 0.5, 'Source L1', transform=axis.transAxes, rotation=90, ha='center', va='center', fontsize=12)

    colorbar = figure.colorbar(image, ax=axis, fraction=0.044, pad=0.05)
    colorbar.ax.tick_params(labelsize=9)
    colorbar.set_label(colorbar_label, fontsize=11, labelpad=14)
    figure.text(0.5, 0.032, footer, ha='center', va='center', fontsize=8.6, wrap=True)

    for extension in OUTPUT_FORMATS:
        figure.savefig(OUTPUT_DIR / f'{filename}.{extension}', dpi=DPI, bbox_inches='tight', facecolor='white')
    if SHOW_FIGURES:
        plt.show()
    plt.close(figure)

## Generate all ten heatmaps

The output directory receives two SVG/PNG layouts for each of: raw attention,
attention budget share, raw absolute message magnitude, message budget share, and
signed gradient × attention.  A cell CSV accompanies each layout.  To change the
output location, formats, labels, or interactive display, edit the first code cell.

In [ ]:
SOURCES = {
    'attention_raw': {
        'sparse': DATA_DIR / 'direct' / 'root_direct_attention_sparse_by_seed.csv.gz',
        'value_column': 'attention_mass',
        'budget_roster': None,
        'budget_column': None,
        'filter_score': False,
        'title': 'RGAT one-hop: raw attention mass',
        'colorbar': 'Conditional mean attention mass α',
        'footer': 'Rows first merge source-visibility groups within a labeled root, then average pair-present roots and the three seeds. — = no matching cell.',
        'signed': False,
        'value_format': lambda value: f'{value:.3f}',
        'metric_name': 'conditional_mean_attention_mass',
    },
    'message_raw': {
        'sparse': DATA_DIR / 'message_contribution' / 'root_message_contribution_sparse_by_seed.csv.gz',
        'value_column': 'absolute_message_l2_sum',
        'budget_roster': None,
        'budget_column': None,
        'filter_score': False,
        'title': 'RGAT one-hop: raw absolute message magnitude',
        'colorbar': 'Conditional mean Σ‖α·Wᵣh‖₂',
        'footer': 'Raw message values sum edge L2 magnitudes, so opposing message directions do not cancel. Rows merge visibility groups, then average pair-present roots and seeds.',
        'signed': False,
        'value_format': lambda value: f'{value:.2f}',
        'metric_name': 'conditional_mean_absolute_message_l2_sum',
    },
    'gradient_x_attention': {
        'sparse': DATA_DIR / 'gradient_x_attention' / 'root_gradient_x_attention_sparse_by_seed.csv.gz',
        'value_column': 'gradient_x_attention',
        'budget_roster': None,
        'budget_column': None,
        'filter_score': True,
        'title': 'RGAT one-hop: L1-pair gradient × attention',
        'colorbar': 'Conditional mean α × ∂ margin / ∂ α',
        'footer': 'Positive values support the predicted-margin score; negative values oppose it. Rows merge visibility groups, then average pair-present roots and seeds.',
        'signed': True,
        'value_format': lambda value: f'{value:+.3f}',
        'metric_name': 'conditional_mean_gradient_x_attention',
    },
}

BUDGET_VIEWS = {
    'attention_budget_share': {
        'source': 'attention_raw',
        'roster': DATA_DIR / 'direct' / 'root_attention_roster_by_seed.csv.gz',
        'budget_column': 'total_attention_mass',
        'title': 'RGAT one-hop: attention budget share',
        'colorbar': 'Share of complete attention budget (%)',
        'footer': 'Each cell is its all-root attention mass divided by the explicit complete target-L1 attention budget, computed per seed then equally averaged. L1-unlabeled sources are outside the displayed cells.',
        'metric_name': 'attention_budget_share_percent',
    },
    'message_budget_share': {
        'source': 'message_raw',
        'roster': DATA_DIR / 'message_contribution' / 'root_message_contribution_roster_by_seed.csv.gz',
        'budget_column': 'typed_absolute_message_l2_sum',
        'title': 'RGAT one-hop: absolute message-magnitude budget share',
        'colorbar': 'Share of complete message-magnitude budget (%)',
        'footer': 'Each cell is its all-root Σ‖α·Wᵣh‖₂ divided by the explicit complete target-L1 message budget, computed per seed then equally averaged. L1-unlabeled sources are outside the displayed cells.',
        'metric_name': 'absolute_message_budget_share_percent',
    },
}

aggregates = {}
for source_name, spec in SOURCES.items():
    print(f'Aggregating {source_name} …')
    raw_totals, raw_counts, budget_numerators = aggregate_sparse(
        spec['sparse'], spec['value_column'], filter_score=spec['filter_score']
    )
    raw_exact, raw_family = raw_cells(raw_totals, raw_counts)
    aggregates[source_name] = {
        'raw_totals': raw_totals,
        'raw_counts': raw_counts,
        'budget_numerators': budget_numerators,
        'raw_exact': raw_exact,
        'raw_family': raw_family,
    }

    for layout, cells, suffix in (
        ('exact_kinship', raw_exact, 'exact_kinship'),
        ('relation_family', raw_family, 'relation_family'),
    ):
        cells_dataframe(cells, layout, spec['metric_name']).to_csv(
            OUTPUT_DIR / f'{source_name}_{suffix}_cells.csv', index=False
        )
        layout_title = 'for exact kinship relations' if layout == 'exact_kinship' else 'by relation family'
        plot_heatmap(
            cells,
            layout=layout,
            title=f"{spec['title']} {layout_title}",
            colorbar_label=spec['colorbar'],
            footer=spec['footer'],
            filename=f'{source_name}_{suffix}',
            signed=spec['signed'],
            value_format=spec['value_format'],
        )

for view_name, view in BUDGET_VIEWS.items():
    print(f'Computing {view_name} …')
    source = aggregates[view['source']]
    totals = load_budget_totals(view['roster'], view['budget_column'])
    exact, family = budget_share_cells(source['budget_numerators'], totals)
    for layout, cells, suffix in (
        ('exact_kinship', exact, 'exact_kinship'),
        ('relation_family', family, 'relation_family'),
    ):
        cells_dataframe(cells, layout, view['metric_name']).to_csv(
            OUTPUT_DIR / f'{view_name}_{suffix}_cells.csv', index=False
        )
        layout_title = 'for exact kinship relations' if layout == 'exact_kinship' else 'by relation family'
        plot_heatmap(
            cells,
            layout=layout,
            title=f"{view['title']} {layout_title}",
            colorbar_label=view['colorbar'],
            footer=view['footer'],
            filename=f'{view_name}_{suffix}',
            signed=False,
            value_format=lambda value: f'{value:.2f}%',
        )

print(f'Wrote {len(list(OUTPUT_DIR.glob("*.png")))} PNG heatmaps, matching SVGs, and cell CSVs to:\n{OUTPUT_DIR}')